# Experiment 4 — Jev + Gemini

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nipundavid/ai-experiments/blob/main/jev/src/experiment_4_gemini.ipynb)

This notebook reproduces the `experiment_4_gemini.py` workflow: Jev chooses a route, then Gemini answers the user request.

> Set `TYPESAFE_API_KEY` and `GOOGLE_API_KEY` before running this notebook.

## Overview

This experiment separates responsibilities cleanly:

- Jev decides the route or category
- Gemini handles the final generation step

This is a practical pattern for combining a structured decision layer with a generative model.

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

if 'TYPESAFE_API_KEY' not in os.environ:
    raise RuntimeError('Set the TYPESAFE_API_KEY environment variable before running this notebook.')
if 'GOOGLE_API_KEY' not in os.environ:
    raise RuntimeError('Set the GOOGLE_API_KEY environment variable before running this notebook.')

In [ ]:
from typing import TypedDict

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_typesafe import Choice, TypeSafeClassifier
from langgraph.graph import END, START, StateGraph

classifier = TypeSafeClassifier(api_key=os.environ['TYPESAFE_API_KEY'])
llm = ChatGoogleGenerativeAI(
    model=os.getenv('GEMINI_MODEL', 'gemini-3.6-flash'),
    api_key=os.environ['GOOGLE_API_KEY'],
)

class State(TypedDict, total=False):
    query: str
    category: str
    answer: str


def classify_query(state: State):
    response = classifier.invoke(
        {
            'state': state['query'],
            'questions': {
                'category': Choice(
                    instructions='Choose the best route for this query.',
                    criteria={
                        'rag': 'The query is about RAG or retrieval.',
                        'coding': 'The query is about code.',
                        'general': 'The query fits neither category.',
                    },
                )
            },
        }
    )
    return {'category': response.choices['category'].choice}


def ask_gemini(state: State):
    response = llm.invoke(
        f'Answer this user query clearly and concisely:\n\n{state["query"]}'
    )
    content = response.content
    if isinstance(content, list):
        content = ''.join(
            block['text']
            for block in content
            if isinstance(block, dict) and block.get('type') == 'text'
        )
    return {'answer': content}


graph = StateGraph(State)
graph.add_node('classify_query', classify_query)
graph.add_node('ask_gemini', ask_gemini)
graph.add_edge(START, 'classify_query')
graph.add_edge('classify_query', 'ask_gemini')
graph.add_edge('ask_gemini', END)
app = graph.compile()

In [ ]:
result = app.invoke({'query': 'Explain why retrieval can improve an LLM answer.'})
print(f'Jev route: {result["category"]}')
print(f'Gemini answer:\n{result["answer"]}')

## Result interpretation

This notebook demonstrates a common production pattern: a small structured decision model routes the request, and a larger generative model handles the final natural-language response.